# Retrieval Target Filter Experiment

이 노트북은 690 chunk DB 기준 최종 retrieval 세팅(96번)을 baseline으로 두고, target document selection과 redundancy control을 강화한 새 실험 결과를 정리한 파일입니다.

기존 prediction/eval 결과는 덮어쓰지 않았고, 새 결과는 아래 폴더에 분리해서 저장했습니다.

`outputs/retrieval_experiments/target_filter_690_full_v1/`


## 1. 기존 구현 확인

새로 만들기 전에 확인한 기존 구현은 아래와 같습니다. 이미 구현되어 있는 부분은 재구현하지 않고 그대로 재사용했습니다.

| 기능 | 상태 | 위치 |
|---|---|---|
| dense retrieval | 이미 있음 | `src/retriever/dense.py`, `src/pipeline/rag_pipeline.py` |
| hybrid search | 이미 있음 | `src/retriever/hybrid.py` |
| rerank | 이미 있음 | `src/retriever/rerank.py` |
| query decomposition | 이미 있음 | `src/retriever/query_decomposition.py` |
| metadata filtering | 이미 있음 | `src/retriever/metadata_filter.py`, `scripts/generate_eval_predictions.py` |
| document diversity | 이미 있음 | `src/retriever/diversity.py` |
| document scoring | 이미 있음 | `src/retriever/document_score.py` |
| target-aware selection | 이미 있음 | `src/retriever/target_aware.py` |
| Chroma / FAISS | 이미 있음 | `src/vectorstore/chroma_store.py`, `src/vectorstore/faiss_store.py` |

새 실험 파일은 기존 `RAGPipeline`을 후보 생성기로 사용하고, 그 위에서만 실험용 target/entity scoring과 redundancy control을 적용합니다.


## 2. 새 실험 파일

실험 스크립트:

`experiments/retrieval_target_filter_experiment.py`

실험 출력:

`outputs/retrieval_experiments/target_filter_690_full_v1/`

생성 파일:

- `A_current_96_baseline.jsonl`
- `B_entity_match_score.jsonl`
- `C_entity_score_redundancy_control.jsonl`
- `D_type_weighted_target_filter.jsonl`
- `E_conservative_preserve3_target_filter.jsonl`
- `retrieval_target_filter_metrics.csv`
- `retrieval_target_filter_per_question_metrics.csv`
- `retrieval_target_filter_best_variant_failure_examples.csv`
- `retrieval_target_filter_summary.md`


## 3. 실험 variant

| variant | 설명 |
|---|---|
| A_current_96_baseline | 기존 96번 결과를 그대로 사용한 baseline |
| B_entity_match_score | 기관명, 사업명, 공고번호, 연도 등 entity match score를 추가 |
| C_entity_score_redundancy_control | B에 기관/사업명 mismatch penalty와 source/evidence 중복 제거 추가 |
| D_type_weighted_target_filter | C에 질문 유형별 키워드 가중치 추가. 이번 실험에서 최고 성능 |
| E_conservative_preserve3_target_filter | 기존 top-3를 보존하고 남은 슬롯만 target filter로 보강 |


## 4. 전체 500문항 결과

| variant_name | hit_at_1 | hit_at_3 | hit_at_5 | mrr_at_5 | ndcg_at_5 | multi_doc_recall_at_5 | target_mismatch_rate_top5 | delta_ndcg_vs_baseline | delta_hit5_vs_baseline | delta_multi_doc_recall_vs_baseline |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| D_type_weighted_target_filter | 0.7082 | 0.9215 | 0.9703 | 0.9467 | 0.9289 | 0.9269 | 0.1071 | 0.0034 | 0.0063 | 0.0156 |
| E_conservative_preserve3_target_filter | 0.7082 | 0.9215 | 0.9693 | 0.9467 | 0.9285 | 0.9245 | 0.1111 | 0.0030 | 0.0053 | 0.0131 |
| A_current_96_baseline | 0.7082 | 0.9215 | 0.9640 | 0.9467 | 0.9255 | 0.9113 | 0.1503 | 0.0000 | 0.0000 | 0.0000 |
| C_entity_score_redundancy_control | 0.7082 | 0.9317 | 0.9603 | 0.9475 | 0.9244 | 0.9023 | 0.0925 | -0.0010 | -0.0037 | -0.0090 |
| B_entity_match_score | 0.6600 | 0.9100 | 0.9450 | 0.9096 | 0.8921 | 0.8645 | 0.0937 | -0.0334 | -0.0190 | -0.0468 |

## 5. 해석

- 96번 baseline도 이미 매우 강해서 단순 entity boost(B)는 오히려 성능을 크게 떨어뜨렸습니다.
- 기관명/사업명만 보고 강하게 재정렬하면 비슷한 기관/비슷한 사업명의 다른 문서가 위로 올라오는 문제가 생겼습니다.
- 반면 질문 유형별 evidence 가중치까지 함께 사용한 D variant는 baseline보다 좋아졌습니다.
- D variant는 `hit@5`, `nDCG@5`, `multi-doc recall@5`가 모두 올랐고, target mismatch rate도 줄었습니다.
- 같은 문서 chunk 반복은 baseline부터 이미 document scoring/target-aware 덕분에 거의 없어서 추가 개선 여지는 작았습니다.

현재 기준으로는 `D_type_weighted_target_filter`가 retrieval 개선 후보입니다.


## 6. 최고 variant 실패 사례 5개

아래는 최고 성능 variant인 D 기준으로도 아직 실패가 남은 사례입니다.

| question_id | variant_name | baseline_hit_at_5 | variant_hit_at_5 | baseline_ndcg_at_5 | variant_ndcg_at_5 | ground_truth_docs | variant_retrieved_docs_top5 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| Q053 | D_type_weighted_target_filter | 0.3333 | 0.3333 | 0.4693 | 0.4693 | ["고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf", "KOICA 전자조달_[긴급] [지문] [국제] 우즈베키스탄 열린 의정활동 상하원 .hwp", "사단법인아시아물위원회사무국_우즈벡-키르기즈스탄 기후변화대응 스.hwp"] | ["고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf", "고려대학교_차세대 포털·학사 정보시스템 구축 사업 재공고.pdf", "고려대학교_고려대학교 공간관리 통합시스템 구축 사업.hwp", "KOICA 전자조달_[지문] 에콰도르 국가 유전자원 데이터 은행 설립 및 역량.hwp", "KOICA 전자조달_[지문] [국 |
| Q054 | D_type_weighted_target_filter | 0.6667 | 0.3333 | 0.6714 | 0.4693 | ["한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp", "나노종합기술원_스마트 팹 서비스 활용체계 구축관련 설비온라인 시스.hwp", "그랜드코리아레저(주)_2024년도 GKL 그룹웨어 시스템 구축 용역.hwp"] | ["한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp", "한국산업기술기획평가원_차세대 NABIS시스템 클라우드 전환·구축 ISP 사.hwp", "한국콘텐츠진흥원_2025~2026년 한국콘텐츠진흥원 정보시스템 통합위탁 운.hwp", "한국의료분쟁조정중재원_차세대 통합정보시스템 구축을 위한 ISP 사업.hwp |
| Q354 | D_type_weighted_target_filter | 0.3333 | 0.3333 | 0.4693 | 0.4693 | ["서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf", "한국철도공사 (용역)_모바일오피스 시스템 고도화 용역(총체 및 1차).hwp", "한국원자력연구원_한국원자력연구원 선량평가시스템 고도화.hwp"] | ["서울시립대학교_[사전공개] 학업성취도 다차원 종단분석 통합시스템 1차.pdf", "서울특별시강동구도시관리공단_(긴급)신규 체육시설 회원관리시스템 구.hwp", "고려대학교_차세대 포털·학사 정보시스템 구축 사업 재공고.pdf", "한국교원대학교_한국교원대학교 AI 에듀테크 센터 통합 플랫폼 구축 정.hwp", "서울특 |
| Q413 | D_type_weighted_target_filter | 0.3333 | 0.3333 | 0.4693 | 0.4693 | ["사단법인아시아물위원회사무국_우즈벡-키르기즈스탄 기후변화대응 스.hwp", "서울특별시 여성가족재단_(재공고, 협상) 서울 디지털성범죄 안심지원센.hwp", "남서울대학교_[혁신-국고] 남서울대학교 스마트 정보시스템 활성화(학사.hwp"] | ["남서울대학교_[혁신-국고] 남서울대학교 스마트 정보시스템 활성화(학사.hwp", "남서울대학교_{대학혁신지원사업] 남서울대학교 대학원 학사시스템 구.hwp", "한국수출입은행_(긴급) SCP 중심의 AML 시스템 구축.pdf", "고려대학교_차세대 포털·학사 정보시스템 구축 사업 재공고.pdf", "인천광역시_2024 |
| Q260 | D_type_weighted_target_filter | 0.5000 | 0.5000 | 0.2641 | 0.2641 | ["한국수자원공사_건설통합시스템(CMS) 고도화.hwp", "고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf"] | ["한국수출입은행_(긴급) 모잠비크 마푸토 지능형교통시스템(ITS) 구축사업.hwp", "KOICA_가나 북부 2개주 포괄적 일차보건의료체계강화사업 수원국시스템활.hwp", "경상북도 영양군_소규모 공공시설 관리시스템 구축사업.hwp", "한국수자원공사_건설통합시스템(CMS) 고도화.hwp", "해외건설협회_에티오피아 부 |

In [ ]:
from pathlib import Path
import pandas as pd

RUN_DIR = Path('../outputs/retrieval_experiments/target_filter_690_full_v1')
metrics = pd.read_csv(RUN_DIR / 'retrieval_target_filter_metrics.csv')
per_question = pd.read_csv(RUN_DIR / 'retrieval_target_filter_per_question_metrics.csv')
failures = pd.read_csv(RUN_DIR / 'retrieval_target_filter_best_variant_failure_examples.csv')

metrics


In [ ]:
# 특정 variant의 실패 문제를 보고 싶을 때
variant_name = 'D_type_weighted_target_filter'
cols = [
    'question_id', 'type', 'difficulty', 'question_type', 'hit_at_5', 'ndcg_at_5',
    'multi_doc_recall_at_5', 'retrieved_docs_top5', 'ground_truth_docs'
]
view = per_question[(per_question['variant_name'] == variant_name) & (per_question['ndcg_at_5'] < 1)].copy()
view[cols].head(20)
